<a href="https://colab.research.google.com/github/Pavangoud-git/Web-Scraping/blob/main/Web_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4 pandas lxml

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
# Scrape All Product URLs........


base_url = "https://books.toscrape.com/catalogue/page-{}.html"

product_urls = []

for page in range(1, 51):

    url = base_url.format(page)

    response = requests.get(url)

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    books = soup.find_all(
        "article",
        class_="product_pod"
    )

    for book in books:

        href = book.h3.a["href"]

        product_url = (
            "https://books.toscrape.com/catalogue/"
            + href
        )

        product_urls.append(product_url)

print("Total URLs:", len(product_urls))

Total URLs: 1000


In [ ]:
# Scrape Detailed Information........

data = []

for idx, url in enumerate(product_urls, start=1):

    response = requests.get(url)

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    title = soup.find("h1").text

    description = ""

    try:
        description = soup.find_all("p")[3].text
    except:
        pass

    category = soup.find_all("a")[3].text

    table = soup.find(
        "table",
        class_="table table-striped"
    )

    rows = table.find_all("tr")

    details = {}

    for row in rows:

        key = row.th.text

        value = row.td.text

        details[key] = value

    rating = soup.find(
        "p",
        class_="star-rating"
    )["class"][1]

    data.append([
        idx,
        title,
        details.get("UPC"),
        details.get("Product Type"),
        details.get("Price (excl. tax)"),
        details.get("Price (incl. tax)"),
        details.get("Tax"),
        details.get("Availability"),
        details.get("Number of reviews"),
        rating,
        category,
        description,
        url
    ])

    time.sleep(0.5)

In [ ]:
# Create DataFrame...


df = pd.DataFrame(
    data,
    columns=[
        "Book_ID",
        "Title",
        "UPC",
        "Product_Type",
        "Price_Excl_Tax",
        "Price_Incl_Tax",
        "Tax",
        "Availability",
        "Number_of_Reviews",
        "Rating",
        "Category",
        "Description",
        "Product_URL"
    ]
)

In [ ]:
# Save CSV.......

df.to_csv(
    "books_complete_dataset.csv",
    index=False
)

print(df.shape)

(1000, 13)
